# Проект: предсказания победителя в онлайн-игре

## Первый этап

In [8]:
import time

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_rows", None)  # or a specific number
pd.set_option("display.max_columns", None)  # to show all columns
pd.set_option("display.expand_frame_repr", False)  # to allow wider DataFrame display

In [9]:
features = pd.read_csv("features/features.csv", index_col='match_id')

In [10]:
nafeat = features.isna().any()
print(*list(nafeat[nafeat].index), sep=", ")

first_blood_time, first_blood_team, first_blood_player1, first_blood_player2, radiant_bottle_time, radiant_courier_time, radiant_flying_courier_time, radiant_first_ward_time, dire_bottle_time, dire_courier_time, dire_flying_courier_time, dire_first_ward_time


1. Какие признаки имеют пропуски среди своих значений? Что могут означать пропуски в этих признаках (ответьте на этот вопрос для двух любых признаков)?

ответ: first_blood_time, first_blood_team, first_blood_player1, first_blood_player2, radiant_bottle_time, radiant_courier_time, radiant_flying_courier_time, radiant_first_ward_time, dire_bottle_time, dire_courier_time, dire_flying_courier_time, dire_first_ward_time

first_blood_time == NaN можно встретить напрмер в партии с индексом 3. Ни один из игроков не совершил ни одного убийства поэтому ивента first_blood не произошло. Соответственно нет и команды которая совершила первое убийство в партии (first_blood_team).

2. Как называется столбец, содержащий целевую переменную?

ответ: radiant_win

In [11]:
target_col = "radiant_win"
features_to_remove = [
    "duration",
    "tower_status_radiant",
    "tower_status_dire",
    "barracks_status_dire",
    "barracks_status_radiant",
]
y = features[target_col].copy()
X = features.drop(features_to_remove + [target_col], axis=1)
X = X.fillna(0)

In [12]:
scaler = StandardScaler()

In [15]:
def run_gradient_boosting(X, y, n_estimators=30):
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    score, elapsed_time = [], []
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        gbc = GradientBoostingClassifier(n_estimators=n_estimators, random_state=241)
        gbc.fit(X[train_ind], y[train_ind])
        y_pred_proba = gbc.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        elapsed_time.append(int(time.time() - start))
        score.append(auc_roc)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).sum()

In [16]:
for n_estimators in [10, 20, 30, 50]:
    score_mean, score_std, elapsed_time_total = run_gradient_boosting(scaler.fit_transform(X), y.to_numpy(), n_estimators)
    print(f"n_estimators={n_estimators} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_total} sec")

n_estimators=10 | AUC-ROC (mean, std): 0.6643877206345741, 0.004902984301311474 | mean time: 45 sec
n_estimators=20 | AUC-ROC (mean, std): 0.6828529377017781, 0.005006424012731842 | mean time: 86 sec
n_estimators=30 | AUC-ROC (mean, std): 0.6894967456506673, 0.004528912040751085 | mean time: 130 sec
n_estimators=50 | AUC-ROC (mean, std): 0.6974540046869849, 0.003911541855296437 | mean time: 219 sec


3. Как долго проводилась кросс-валидация для градиентного бустинга с 30 деревьями? Инструкцию по измерению времени можно найти ниже по тексту. Какое качество при этом получилось? Напомним, что в данном задании мы используем метрику качества AUC-ROC.

ответ: Время для кросс-валидации с 30 деревьями составило ~= 130s с качеством AUC-ROC ~= 0.69

4. Имеет ли смысл использовать больше 30 деревьев в градиентном бустинге? Что бы вы предложили делать, чтобы ускорить его обучение при увеличении количества деревьев?
   
ответ: При увеличение числа деревьев с 30 до 50 качество выростло с 0.69 до 0.70, минусом является увеличение так же времени обучения. Если время обучения остается удовлетворительным, то да, стоит. Для ускорения обучения при увеличении количества деревьев можно уменьшить их размер, параметром max_depth или использовать раннюю остановку мониторя валидационный лосс.

## Второй этап

In [25]:
def run_logistic_regression(X, y, C=1.0):
    score, elapsed_time = [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        gbc = LogisticRegression(
            random_state=241,
            penalty="l2",
            C=C,
            max_iter=1000
        )
        gbc.fit(X[train_ind], y[train_ind])
        y_pred_proba = gbc.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        score.append(auc_roc)
        elapsed_time.append(time.time() - start)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).sum()

In [26]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | total time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 3.38 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 3.91 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.28 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.16 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.30 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.67 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.39 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7163, 0.0028 | total time: 4.50 sec


In [27]:
X_without_cat = X.drop(
    [
        "lobby_type",
        "r1_hero",
        "r2_hero",
        "r3_hero",
        "r4_hero",
        "r5_hero",
        "d1_hero",
        "d2_hero",
        "d3_hero",
        "d4_hero",
        "d5_hero",
    ], axis=1
)

In [32]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X_without_cat), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | mean time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7162, 0.0028 | mean time: 3.47 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.23 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.74 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 5.18 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.57 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.20 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.05 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7164, 0.0029 | mean time: 4.13 sec


In [33]:
unique_heros = (
    set(features["r1_hero"].unique())
    | set(features["r2_hero"].unique())
    | set(features["r3_hero"].unique())
    | set(features["r4_hero"].unique())
    | set(features["r5_hero"].unique())
    | set(features["d1_hero"].unique())
    | set(features["d2_hero"].unique())
    | set(features["d3_hero"].unique())
    | set(features["d4_hero"].unique())
    | set(features["d5_hero"].unique())
)

In [34]:
print(f"Number of heroes in game: {max(unique_heros)}")
print(f"Number of unique heroes in dataset: {len(unique_heros)}")

Number of heroes in game: 112
Number of unique heroes in dataset: 108


In [35]:
X_pick = np.zeros((len(features), max(unique_heros)))
for i, match_id in enumerate(features.index):
    for p in range(1, 6):
        X_pick[i, int(features.at[match_id, f"r{p}_hero"])-1]  = 1
        X_pick[i, int(features.at[match_id, f"d{p}_hero"])-1]  = -1

In [36]:
X_pick = pd.DataFrame(X_pick, index=features.index, columns=[f"hero_{i+1}" for i in range(max(unique_heros))])

In [37]:
X_encoded_cat = pd.concat([X_without_cat, X_pick], axis=1)

In [38]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X_encoded_cat), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | mean time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7517, 0.0023 | mean time: 4.13 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 5.89 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 6.93 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7519, 0.0020 | mean time: 6.63 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 6.39 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 6.43 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 5.99 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7519, 0.0021 | mean time: 5.81 sec
